# Lab 9b: Sequence Labelling with BERT
In this bonus lab, we will look at sequence labelling with BERT.

It is heavily based on the huggingface tutorial: https://huggingface.co/learn/nlp-course/en/chapter7/2

## Data Preparation

First of all we need a dataset suitable for sequence labelling.  Heere we use the CoNLL-2003 dataset, which contains news stories from Reuters which have been annotated for part-of-speech and named entities.

In [ ]:
#needed on colab
#!pip install datasets

In [ ]:
from datasets import load_dataset
raw_datasets=load_dataset("conll2003",trust_remote_code=True)

Inspecting the raw_datasets object shows is the columns present and the split between training, validation and set.

In [ ]:
raw_datasets

Let's have a look at the first element of the training set in terms of tokens and ner_tags.

In [ ]:
raw_datasets["train"][0]["tokens"]

In [ ]:
raw_datasets["train"][0]["ner_tags"]

To see what these integer labels actually mean, we need to refer to the features of the dataset and their names

In [ ]:
ner_feature = raw_datasets["train"].features["ner_tags"]
label_names=ner_feature.feature.names
label_names

### Exercise 1
Write a function to which takes an index to the training data  as input and displays the corresponding sentence tokens and label_names as in this example:

```
display_labels(index=0)
--------------------------
EU : B-ORG
rejects : O
German : B-MISC
call : O
to : O
boycott : O
British : B-MISC
lamb : O
. : O
```


## Tokenisation

Now we are going to need to tokenize the text and then align the labels.  
We need to use the standard BERT tokenizer but tell it that the text is already split into words.  It will then carry out wordpiece tokenization.  We then need to align the labels - which requires adding labels where a word in the original text has been split into multiple wordpieces.

In [ ]:
from transformers import AutoTokenizer
model_checkpoint="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
#lets try the tokenizer on different lines of the training data
#find a line where at least one word is split into subwords
inputs=tokenizer(raw_datasets["train"][3]["tokens"],is_split_into_words=True)
inputs.tokens()

If we look at the word_ids for this sentence, we can see that special tokens (CLS and SEP) do not have a word_id (as they are not in the vocabulary).  A word which has been split into multiple subwords has the same word_id for each subword.

In [ ]:
inputs.word_ids()

### Exercise2
We need a function to align the labels with the new tokens.  In particular, we need to use a special label '-100'  for any tokens which are not in the vocabulary (i.e., don't have a word_id) and create the label for subsequent parts of a word. If the label is B-XXX then the subsequenct label needs to be I-XXX.  Make sure you understand the following function which takes a list of labels and a list of word_ids and returns a new *aligned* list of labels.  **Test it out** on some different inputs from the training data

In [ ]:
def align_labels_with_tokens(labels,word_ids):
    new_labels=[]
    current_word = ''
    for word_id in word_ids:
        if word_id ==current_word:
            #same word as previous token
            label = labels[word_id]
            #update B-XXX to I-XXX
            if label % 2==1:
                label+=1
        else:
            #start of new word
            current_word=word_id
            if word_id is None:
                #special token
                label=-100

            else:
                label=labels[word_id]

        new_labels.append(label)

    return new_labels

In [1]:
# test out the align_labels_with_tokens() function

We can now apply this to the whole dataset

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

In [ ]:
tokenized_datasets

In [ ]:
tokenized_datasets["train"][0]["input_ids"]

In [ ]:
tokenized_datasets["train"][0]["labels"]

## Adding padding

We need to make sure all of the inputs in a batch are the same length.  We do this by adding padding using the DataCollatorForTokenClassification class.

In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)

#let's see what this looks like for a batch of size 2 (the first 2 items in the training data)
batch=data_collator([tokenized_datasets["train"][i] for i in range(2)])
batch["labels"]

## Evaluation Metrics
We need to be able to evaluate how good the predictions of a model are.

### Exercise 3
Write a function compute_accuracy which will take predictions and references (as lists of lists of labels) and return the accuracy.

For example:

```
compute_accuracy([['O','O','B-PER','O'],['B-MISC','I-MISC','O']],[['O','O','B-PER','O'],['B-MISC','I-MISC','O']])
-----------------
0.857
```

Now we are going to apply this to the logits which will come back from the model.  We need to select the most likely prediction for each token using argmax and then strip away special tokens before passing the lists of lists of labels to the compute_accuracy function just defined.

In future, we could define (or import) a greater range of evaluation metrics (e.g., precision and recall for each of the different labels), but here we are just going to use overall accuracy as an indicator of how well our model might be doing.  You might want to check out the seqeval library

In [ ]:

label_names

In [ ]:
import numpy as np

def compute_metrics(eval_preds):
    #eval_preds is a batch of logits and labels (list of lists)
    logits,labels = eval_preds
    predictions=np.argmax(logits, axis=-1)

    #remove ignored index (special tokens) and convert to labels
    true_labels=[[label_names[l] for l in label if l!=-100] for label in labels]
    true_predictions = [[label_names[p] for (p,l) in zip(prediction,label) if l!=-100] for prediction,label in zip(predictions,labels)]

    #we could compute other metrics as well, but here, for simplicity we are just going to compute overall accuracy
    accuracy = compute_accuracy(predictions=true_predictions,references=true_labels)
    return {"overall accuracy":accuracy}


## Defining the model

First of all we need to make sure we have 2 dictionaries which define the correspondance between ids and labels

In [ ]:
id2label={i:label for i,label in enumerate(label_names)}
label2id={v:k for k,v in id2label.items()}

print(id2label)
print(label2id)

In [ ]:
model_checkpoint

Now we can set up the pre-trained model ready to be used for sequence labelling / token classification

In [ ]:
from transformers import AutoModelForTokenClassification

model =AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id)

In [ ]:
model.config.num_labels

In [ ]:
#I cannot run this on my laptop.  But if I train the model on colab, I can save it, download it and then run it on my laptop
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
)

In [ ]:
#make sure you have changed the runtype to GPU on colab!

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)
trainer.train()

In [ ]:
#once the model has been saved, it can be run independently elsewhere
trainer.save_model("julie-ner")


## Using the Model

The saved model can now be used in a number of ways.  It can be used as part of a "token-classification" pipeline which automatically carries out the appropriate tokenization on the input and then runs the model

In [ ]:
#run this without training (or running any of the previous cells) if the model is saved (and available)

from transformers import pipeline

ner_classifier=pipeline("token-classification",model="julie-ner")

sentence="Germans shun British lamb"
ner_classifier(sentence)

Alternatively, the model can be run directly on tokenized inputs, such as we have already prepared for our test data.

The cells below will remind us what those datasets look like - both in raw and tokenized forms.

In [ ]:
raw_datasets["test"][0]

In [ ]:
tokenized_datasets["test"][0]

We can instantiate an AutoModelForTokenClassification model using the saved checkpoint.

In [ ]:
from transformers import AutoModelForTokenClassification
model_checkpoint="julie-ner"
model =AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id)

Now we need to wrap the attention_mask and input_ids as tensors and pass that to the model.


In [ ]:
import torch
test_input=tokenized_datasets["test"][0]
mask_tensor=torch.tensor([test_input['attention_mask']])
input_id_tensor = torch.tensor([test_input['input_ids']]).squeeze(1)


# If you have a GPU, put everything on cuda
#input_id_tensor = input_id_tensor.to('cuda')
#mask_tensor = mask_tensor.to('cuda')
#model.to('cuda')

output=model(input_id_tensor,mask_tensor)
output[0]

We need to extract the logits from the output tensor

In [ ]:
logits=output[0][0].detach().numpy()
logits

Then we can use numpy to get the most likely label according to the logits

In [ ]:
predictions=np.argmax(logits, axis=-1)
predictions

### Exercise 4
Write code to display the input id, true label id and predicted label id with one token per line e.g.:

```
101 : -100 : 0
4715 : 0 : 0
1011 : 0 : 0
2900 : 5 : 5
....
```

Can you improve this so that special tokens are removed and so that it shows the actual token, the actual true label and the actual predicted label e.g.:

```
soccer : O : O
- : O : O
japan : B-LOC : B-LOC
```



### Exercise 5
Write some code which will run a list of sentences (e.g., from the test data) through the named entity recogniser and collect the predictions.  Compute the overall accuracy of the model on the test data

### Extension
Compute precision, recall and F1 for each of the different named entity labels.  Compute micro and macro-average F1 scores.